# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices (Northern Kenya) Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration and processing of the FAIR⁲ dataset using the [mlcroissant](https://pypi.org/project/mlcroissant/) library. All dataset entities (record sets, fields, columns) are referenced by their `@id` according to the Croissant specification.

### Dataset Source
The dataset schema is published [at this Croissant JSON-LD URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and dataset records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load Dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Returned as a DatasetMetadata object

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview

Here we retrieve and review the available record set `@id`s, and for each, its fields and columns. Knowing the `@id` enables precise, schema-compliant data referencing in later steps.

In [ ]:
# List all record sets with their @id and associated field @ids
record_sets = list(dataset.record_sets.keys())
print('Available record set @ids:')
for rs_id in record_sets:
    rs = dataset.record_sets[rs_id]
    print(f'\n- Record Set @id: {rs_id}')
    fields = getattr(rs, 'fields', [])
    if fields:
        print(f'  Fields @id: {[f["@id"] for f in fields]}')
    else:
        print('  No fields listed. Attempting to infer from data...')

# For demonstration, display a preview of records for the first (if available)
if record_sets:
    example_record_set = record_sets[0]
    print(f'\nFirst five records from record set {example_record_set}:')
    for i, rec in enumerate(dataset.records(record_set=example_record_set)):
        if i>=5:
            break
        print(rec)
else:
    print('No record sets found in schema.')


## 3. Data Extraction

Extract all records from each available record set (by `@id`) into pandas DataFrames. 

You may specify the particular record set(s) you want to analyze by their `@id`. All fields remain referenced by their Croissant entity `@id`s.

In [ ]:
# Extract each available record set into a pandas DataFrame (by @id)
dataframes = {}
for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set @id: {rs_id}")
        print(f"Columns (@id): {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records loaded for record set @id: {rs_id}")

# Select one record set for further exploratory data analysis (EDA)
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]  # Or set explicitly to your main interest
    print(f"Main record set for EDA: {main_record_set_id}")
else:
    main_record_set_id = None
    print('No dataframes extracted.')


## 4. Exploratory Data Analysis (EDA)

In this section, we demonstrate several common data processing steps including filtering, normalization, and grouping, all referencing field/column `@id`.

- **Filtering**: Select records where a chosen numeric field (@id) exceeds a threshold.
- **Normalization**: Standardize this numeric field.
- **Grouping**: Aggregate statistics by a categorical variable (by `@id`).

In [ ]:
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print(f"Available columns: {df.columns.tolist()}")

    # Heuristically select the first numeric column for demo EDA
    numeric_field_id = None
    for c in df.columns:
        if pd.api.types.is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
    if numeric_field_id is None:
        # Try to coerce any floatable column
        for c in df.columns:
            try:
                pd.to_numeric(df[c], errors='raise')
                numeric_field_id = c
                df[c] = pd.to_numeric(df[c], errors='coerce')
                break
            except Exception:
                continue
    if numeric_field_id is None:
        print('No numeric fields found in DataFrame.')
    else:
        print(f"Using numeric field @id for demo: {numeric_field_id}")
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean):")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Choose a potential group field (categorical; not numeric field itself)
        group_field_id = None
        for c in df.columns:
            if c != numeric_field_id and df[c].nunique() > 1 and df[c].nunique()<df.shape[0]//2:
                group_field_id = c
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
        else:
            print('No suitable categorical group field detected.')
else:
    print('No data available for EDA.')


## 5. Visualization

Visualize data distributions and variable relationships using `matplotlib` or `seaborn`. Update the plot below to use your selected `@id` field(s) of interest.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    # If group field available, show group means
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind='bar')
        plt.title(f"{numeric_field_id} mean by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()
else:
    print('Not enough numeric data for visualization.')


## 6. Conclusion

In this notebook, we demonstrated how to load, explore, and process a Croissant-based dataset using the `mlcroissant` Python library:

- All fields and records were referenced **by their `@id`**, as recommended by the Croissant specification.
- We reviewed available record sets and their structure, inspected records, and loaded them into tidy pandas DataFrames.
- Basic EDA (filtering, normalization, grouping) and simple visualizations illustrated various schema-aware exploratory workflows for research analysis.

For further analysis, you can explore additional fields, more advanced statistics, and customize visualizations to your needs. Refer to the [mlcroissant documentation](https://mlcroissant.org/) for more advanced functionalities.